In [48]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(palette="rocket")

### Data Dictionary: Auction Lead Analysis

| Column name | Data type | Description |
| :--- | :--- | :--- |
| **auction_id** | `int64` | Unique identifier for each specific auction event. |
| **bid** | `float64` | The value of the specific bid entered by the bidder. |
| **bid_time** | `float64` | Time elapsed (in days) since the auction started. |
| **bidder** | `object` | Anonymized username or ID of the person placing the bid. |
| **bidder_rate**| `float64` | The feedback rating of the bidder (proxy for experience). |
| **open_bid** | `float64` | The starting price set by the seller. |
| **hammer_price** | `float64` | The final selling price of the item (the winning bid). |
| **item** | `object` | Category or name of the object being auctioned (e.g., Cartier wristwatch). |
| **auction_type** | `object` | The duration of the auction (e.g., "3 day auction"). |
| **duration_days** | `int/float`| **(Engineered)** The total length of the auction extracted as a numeric value. |
| **relative_bid_time**| `float64` | **(Engineered)** Bid time as a % of total duration (0.0 = start, 1.0 = end). |
| **bid_increment** | `float64` | **(Engineered)** The "jump" in price from the previous bid in that auction. |
| **realization_ratio**| `float64` | **(Engineered)** The ratio of hammer_price to opening_price (Market Heat). |
| **is_winner** | `bool` | **(Engineered)** True if this specific row represents the winning bid. |
| **is_snipe** | `bool` | **(Engineered)** True if the bid was placed in the final 5% of the auction time. |

In [49]:
auction_df = pd.read_csv("../data/auction.csv")

auction_df.head()

,auctionid,bid,bidtime,bidder,bidderrate,openbid,price,item,auction_type
0,1638893549,175.0,2.230949,schadenfreud,0.0,99.0,177.5,Cartier wristwatch,3 day auction
1,1638893549,100.0,2.600116,chuik,0.0,99.0,177.5,Cartier wristwatch,3 day auction
2,1638893549,120.0,2.600810,kiwisstuff,2.0,99.0,177.5,Cartier wristwatch,3 day auction
3,1638893549,150.0,2.601076,kiwisstuff,2.0,99.0,177.5,Cartier wristwatch,3 day auction
4,1638893549,177.5,2.909826,eli.flint@flightsafety.co,4.0,99.0,177.5,Cartier wristwatch,3 day auction


In [50]:
swarovski_df = pd.read_csv("../data/swarovski.csv")

swarovski_df.head()

,Seller,Bidder,Weight,Bidder.Volume,Seller.Volume
0,332874919,718577508,2,3,547
1,594667804,399983466,5,6,183
2,663070601,655828811,1,4,274
3,309608641,599835541,3,8,3986
4,201729374,693022555,1,2,4681


In [51]:
print(auction_df.shape)
auction_df.dtypes

(10681, 9)


auctionid         int64
bid             float64
bidtime         float64
bidder              str
bidderrate      float64
openbid         float64
price           float64
item                str
auction_type        str
dtype: object

In [52]:
print(f"Total duplicates: {auction_df.duplicated().sum()}")

print("\nTotal missing values:\n\n", auction_df.isna().sum())

Total duplicates: 0

Total missing values:

 auctionid        0
bid              0
bidtime          0
bidder          16
bidderrate      11
openbid          0
price            0
item             0
auction_type     0
dtype: int64


In [53]:
auction_df_clean = auction_df.copy()

auction_df_clean["bidder"] = auction_df_clean["bidder"].fillna("Unknown")

# Using 0 for bidderrate (assuming new/guest user)
auction_df_clean["bidderrate"] = auction_df_clean["bidderrate"].fillna(0)

print("Total missing values:\n\n", auction_df_clean.isna().sum())

Total missing values:

 auctionid       0
bid             0
bidtime         0
bidder          0
bidderrate      0
openbid         0
price           0
item            0
auction_type    0
dtype: int64


In [54]:
auction_df_clean = auction_df_clean.rename(columns = {
    "auctionid": "auction_id",
    "bidtime": "bid_time",
    "bidderrate": "bidder_rate", 
    "openbid": "open_bid",
    "price": "hammer_price"
})

auction_df_clean.head()

,auction_id,bid,bid_time,bidder,bidder_rate,open_bid,hammer_price,item,auction_type
0,1638893549,175.0,2.230949,schadenfreud,0.0,99.0,177.5,Cartier wristwatch,3 day auction
1,1638893549,100.0,2.600116,chuik,0.0,99.0,177.5,Cartier wristwatch,3 day auction
2,1638893549,120.0,2.600810,kiwisstuff,2.0,99.0,177.5,Cartier wristwatch,3 day auction
3,1638893549,150.0,2.601076,kiwisstuff,2.0,99.0,177.5,Cartier wristwatch,3 day auction
4,1638893549,177.5,2.909826,eli.flint@flightsafety.co,4.0,99.0,177.5,Cartier wristwatch,3 day auction


In [55]:
print(auction_df_clean["item"].unique())
print(auction_df_clean["auction_type"].unique())

<StringArray>
['Cartier wristwatch', 'Palm Pilot M515 PDA', 'Xbox game console']
Length: 3, dtype: str
<StringArray>
['3 day auction', '5 day auction', '7 day auction']
Length: 3, dtype: str


In [56]:
auction_df_clean["auction_duration_days"] = auction_df_clean["auction_type"].str.extract("(\d+)").astype(int)

auction_df_clean["relative_bid_time"] = auction_df_clean["bid_time"] / auction_df_clean["auction_duration_days"]

auction_df_clean.head(3)

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_22009/603796157.py:1: SyntaxWarning: invalid escape sequence '\d'
  auction_df_clean["auction_duration_days"] = auction_df_clean["auction_type"].str.extract("(\d+)").astype(int)


,auction_id,bid,bid_time,bidder,bidder_rate,open_bid,hammer_price,item,auction_type,auction_duration_days,relative_bid_time
0,1638893549,175.0,2.230949,schadenfreud,0.0,99.0,177.5,Cartier wristwatch,3 day auction,3,0.743650
1,1638893549,100.0,2.600116,chuik,0.0,99.0,177.5,Cartier wristwatch,3 day auction,3,0.866705
2,1638893549,120.0,2.600810,kiwisstuff,2.0,99.0,177.5,Cartier wristwatch,3 day auction,3,0.866937


In [57]:
# Adding bid momentum for each auction
auction_df_clean = auction_df_clean.sort_values(["auction_id", "bid_time"])
auction_df_clean["bid_increment"] = auction_df_clean.groupby("auction_id")["bid"].diff().fillna(0)


# Calculating price realization
auction_df_clean["realization_ratio"] = auction_df_clean["hammer_price"] / auction_df_clean["open_bid"]


# Flagging winner
auction_df_clean["is_winner"] = auction_df_clean["bid"] == auction_df_clean["hammer_price"]


# A bid is a snipe if it happens in the final 5% of the auction 
auction_df_clean["is_snipe"] = auction_df_clean["relative_bid_time"] > 0.95

snipe_percentage = auction_df_clean["is_snipe"].mean() * 100
print(f"Percentage of bids that are 'Snipes': {snipe_percentage:.2f}%")

Percentage of bids that are 'Snipes': 32.10%


In [58]:
auction_df_clean.describe()

,auction_id,bid,bid_time,bidder_rate,open_bid,hammer_price,auction_duration_days,relative_bid_time,bid_increment,realization_ratio
count,1.068100e+04,10681.000000,10681.000000,10681.000000,10681.000000,10681.000000,10681.000000,10681.000000,10681.000000,10681.000000
mean,4.136148e+09,207.586109,3.979628,31.903848,52.246256,335.043589,5.939612,0.670505,11.574572,4645.677826
std,2.489918e+09,323.037396,2.353386,120.536308,168.453245,433.566009,1.584867,0.339510,68.624678,11787.305096
min,1.638844e+09,0.010000,0.000567,-4.000000,0.010000,26.000000,3.000000,0.000081,-1080.000000,1.000000
25%,3.015329e+09,72.000000,1.949931,1.000000,1.000000,186.510000,5.000000,0.361319,0.000000,4.300000
50%,3.020526e+09,140.000000,4.140833,5.000000,4.990000,228.490000,7.000000,0.821879,5.000000,51.051051
75%,8.212136e+09,210.000000,6.448060,21.000000,50.000000,255.000000,7.000000,0.974873,15.000000,374.990000
max,8.215611e+09,5400.000000,6.999990,3140.000000,5000.000000,5400.000000,7.000000,0.999999,1150.000000,160000.000000
